# FRC match videos from The Blue Alliance

Finds YouTube videos of FRC matches through The Blue Alliance (TBA) API and downloads the full videos with `yt-dlp`.

1. Get a free TBA read key at https://www.thebluealliance.com/account (Read API Keys).
2. Fill in the settings cell and run the cells top to bottom.
3. Re-running is safe: videos already downloaded are skipped.

In [ ]:
%pip install -q requests yt-dlp tqdm
# yt-dlp needs ffmpeg to merge video+audio. On a Mac: brew install ffmpeg

In [ ]:
import os, json, time, getpass, subprocess, sys
from pathlib import Path
import requests
from tqdm.auto import tqdm

# ---- settings ----
TBA_KEY = os.environ.get("TBA_KEY") or getpass.getpass("TBA read API key: ")
YEAR = 2026                    # season to pull
EVENT_KEYS = []                # e.g. ["2026txhou", "2026casj"]; leave empty for every event that year
EVENT_TYPES = None             # None = all; or e.g. {0,1,2,3,4} (regional, district, district cmp, cmp division, cmp finals)
COMP_LEVELS = None             # None = all; or e.g. {"qm"} for quals only, {"qf","sf","f"} for playoffs
MAX_VIDEOS = 50                # stop after this many downloads (None = no limit)
QUALITY = "bestvideo[height<=1080]+bestaudio/best[height<=1080]"
OUT = Path("frc_match_videos")   # videos land in OUT/<event_key>/<match_key>.mp4
OUT.mkdir(exist_ok=True)

In [ ]:
BASE = "https://www.thebluealliance.com/api/v3"
S = requests.Session()
S.headers.update({"X-TBA-Auth-Key": TBA_KEY, "User-Agent": "frc-video-notebook"})

def tba(path):
    for attempt in range(5):
        r = S.get(BASE + path, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 401:
            raise SystemExit("TBA rejected the API key (401). Check TBA_KEY.")
        if r.status_code in (429, 500, 502, 503):
            time.sleep(2 ** attempt); continue
        r.raise_for_status()
    raise RuntimeError(f"TBA request kept failing: {path}")

print("Status:", tba("/status").get("current_season"))

In [ ]:
# 1) pick events
if EVENT_KEYS:
    events = [tba(f"/event/{k}/simple") for k in EVENT_KEYS]
else:
    events = tba(f"/events/{YEAR}/simple")
    if EVENT_TYPES is not None:
        events = [e for e in events if e["event_type"] in EVENT_TYPES]
events.sort(key=lambda e: e.get("start_date") or "")
print(len(events), "events")

In [ ]:
# 2) collect every match that has a YouTube video
videos = []
for ev in tqdm(events, desc="events"):
    try:
        matches = tba(f"/event/{ev['key']}/matches")
    except Exception as e:
        print("skip", ev["key"], e); continue
    for m in matches:
        if COMP_LEVELS and m["comp_level"] not in COMP_LEVELS:
            continue
        for v in m.get("videos") or []:
            if v.get("type") == "youtube" and v.get("key"):
                vid = v["key"].split("?")[0].split("&")[0]   # some keys carry ?t= offsets
                videos.append({
                    "event_key": ev["key"], "event_name": ev["name"], "match_key": m["key"],
                    "comp_level": m["comp_level"], "match_number": m["match_number"],
                    "set_number": m["set_number"],
                    "red": m["alliances"]["red"]["team_keys"], "blue": m["alliances"]["blue"]["team_keys"],
                    "youtube_id": vid, "url": f"https://www.youtube.com/watch?v={vid}",
                })
# drop duplicates (one match can list the same video twice)
seen, uniq = set(), []
for v in videos:
    if (v["match_key"], v["youtube_id"]) not in seen:
        seen.add((v["match_key"], v["youtube_id"])); uniq.append(v)
videos = uniq
(OUT / "videos_index.json").write_text(json.dumps(videos, indent=2))
print(len(videos), "match videos found; index saved to", OUT / "videos_index.json")

In [ ]:
# 3) download the full videos
done = failed = 0
todo = videos if MAX_VIDEOS is None else videos[:MAX_VIDEOS]
for v in tqdm(todo, desc="downloading"):
    folder = OUT / v["event_key"]; folder.mkdir(exist_ok=True)
    stem = v["match_key"] + ("" if sum(x["match_key"] == v["match_key"] for x in videos) == 1 else f"_{v['youtube_id']}")
    if list(folder.glob(stem + ".*")) and not list(folder.glob(stem + ".*.part")):
        continue  # already have it
    cmd = [sys.executable, "-m", "yt_dlp", "-f", QUALITY, "--merge-output-format", "mp4",
           "--no-playlist", "--retries", "5", "-o", str(folder / (stem + ".%(ext)s")), v["url"]]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        done += 1
    else:
        failed += 1
        print("failed", v["match_key"], (r.stderr.strip().splitlines() or ["?"])[-1])
    time.sleep(2)  # be polite to YouTube
print(f"downloaded {done}, failed {failed}, saved under {OUT.resolve()}")